# Mechanical CAD editing: multi-source QLoRA + executed geometry + FEM

Fresh Qwen2.5-Coder-3B-Instruct, two epochs, BenchCAD + Microsoft CAD-Editor.
Use **Runtime → Change runtime type → GPU**, then **Run all**. This notebook contains the audited public edit subset and its STEP references; no SSH passwords or API keys.

The model predicts edits. External solvers validate geometry and six explicitly contracted FEM examples. Missing/failed FEM stays not verified. Actual operating safety is not established. No drawing-perception or FEM-reward training is claimed.

In [ ]:
!pip -q install "transformers==4.51.3" "peft==0.15.2" accelerate bitsandbytes "cadquery==2.8.0" "gmsh==4.15.2" meshio scipy
!apt-get -qq update && apt-get -qq install -y libglu1-mesa libgl1
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())

In [ ]:
#@title Unpack audited public CAD edit data and scripts
import base64, hashlib, io, tarfile
from pathlib import Path
# Audited public source package; embedded for a credential-free portable notebook.
encoded = """
...
"""
bundle=base64.b64decode(encoded)
assert hashlib.sha256(bundle).hexdigest()=="703d1c11795a1e4c2138bdfd67df1925679cec138a713a3fc279930b534cefad"
ROOT=Path("/content/cad_edit"); ROOT.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle),mode="r:gz") as t: t.extractall(ROOT,filter="data")
print("Source package verified and extracted")

## Training and before/final evaluation

4-bit NF4 double quantization; BF16 on capable GPUs, FP16 on T4. LoRA rank 16, alpha 320, last 12 blocks, all attention/MLP projections. Effective batch four; 4,096-token context; 768-token generation cap. Final checkpoint only. All retained held-out cases are evaluated and exclusions recorded. No reference edits repair model outputs.

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT=Path("/content/cad_edit")
RUN=ROOT/"results"
if (RUN/"run_manifest.json").exists() and json.loads((RUN/"run_manifest.json").read_text())["status"]=="completed":
    print("Completed run already exists; not training twice")
else:
    assert not RUN.exists(), "Incomplete run exists: inspect before resuming or choose a new output directory"
    process=subprocess.Popen([sys.executable,"-u",str(ROOT/"scripts/train_multisource_cad_colab.py"),"--data",str(ROOT/"data"),"--output",str(RUN)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout: print(line,end="",flush=True)
    assert process.wait()==0, "Training failed; inspect the log"

## Execute both native CAD representations

BenchCAD uses author-released STEP targets. CAD-Editor uses locally reconstructed six-bit sketch/extrude targets; these are not author-released STEP files. Geometry metrics stay separate by dataset.

In [ ]:
for label in ["base","trained"]:
    subprocess.run([sys.executable,"-u",str(ROOT/"scripts/score_multisource_cad_geometry.py"),str(RUN/(label+"-predictions.jsonl"))],check=True)
print(json.loads((RUN/"trained-predictions-geometry.json").read_text()))

## FEM of the NEW adapter

Audit all six illustrated held-out cases. Fixed synthetic E=210,000 MPa, nu=0.3, units mm/N/MPa and 1 N geometric end-band loads. The shaft fixture co-rotates. Invalid patches, unmeshable parts, unanchored components and unconverged responses fail verification. Similar compliance cannot override a wrong solid. Stress is diagnostic; no strength claim.

In [ ]:
subprocess.run([sys.executable,"-u",str(ROOT/"scripts/colab_verify_mechanical_examples.py"),"--root",str(ROOT),"--run",str(RUN)],check=True)

## Save results

The ZIP contains model adapters, manifests, raw predictions, native geometry scores and FEM solver fields. Download before disconnecting; Colab storage is ephemeral. No Google Drive mount is required.

In [ ]:
import zipfile
from google.colab import files
archive=Path("/content/multisource-cad-colab-results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for f in RUN.rglob("*"):
        if f.is_file() and "checkpoints" not in f.parts: z.write(f,f.relative_to(RUN))
print("Completed artifact:",archive)
files.download(str(archive))